In [4]:
import os
import random
import time
import copy
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from tqdm import tqdm

import numpy as np
import itertools
import math
%matplotlib inline 
from matplotlib import pyplot as plt

from ofa.elastic_nn.networks import OFAMobileNetV3
from ofa.imagenet_codebase.utils import cross_entropy_with_label_smoothing, subset_mean, list_mean
from ofa.elastic_nn.utils import set_running_statistics
from ofa.utils import AverageMeter, accuracy
from ofa.imagenet_codebase.data_providers.base_provider import MyRandomResizedCrop
from NAS.imagenet_eval_helper import evaluate_ofa_subnet

In [5]:
cuda_available = torch.cuda.is_available()
if cuda_available:
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True
    print('Using GPU.')
else:
    print('Using CPU.')

Using GPU.


In [6]:
def build_train_transform():
    # image_size = [128, 160, 192, 224]
    image_size = 224
    color_transform = None
    resize_transform_class = transforms.Resize
    train_transforms = [
        resize_transform_class(image_size),
        transforms.RandomHorizontalFlip(),
    ]
    train_transforms.append(transforms.ColorJitter(brightness=32. / 255., saturation=0.5))
    train_transforms += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.45785159, 0.40990421, 0.3922225 ], std=[0.23462605, 0.22015331, 0.23121287]),
    ]
    train_transforms = transforms.Compose(train_transforms)
    return train_transforms

def build_valid_transform():
    image_size = 224
    return transforms.Compose([
            transforms.Resize(int(math.ceil(image_size / 0.875))),
            transforms.CenterCrop(image_size),
            transforms.ColorJitter(brightness=32. / 255., saturation=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.45488905, 0.40866664, 0.38849462], std=[0.23486623, 0.22084754, 0.23113226]),
        ])

train_dataset = ImageFolder('data/train', build_train_transform())
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=28, pin_memory=True)

val_dataset = ImageFolder('data/val', build_valid_transform())
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=28, pin_memory=True)

def build_sub_train_loader(n_images, batch_size, num_worker=None, num_replicas=None, rank=None):
    num_worker = train_loader.num_workers
    n_samples = len(train_loader.dataset.samples)
    g = torch.Generator()
    g.manual_seed(937162211)
    rand_indexes = torch.randperm(n_samples, generator=g).tolist()

    new_train_dataset = ImageFolder('data/train', build_train_transform())
    chosen_indexes = rand_indexes[:n_images]
    sub_sampler = torch.utils.data.sampler.SubsetRandomSampler(chosen_indexes)
    sub_data_loader = torch.utils.data.DataLoader(
        new_train_dataset, batch_size=batch_size, sampler=sub_sampler,
        num_workers=num_worker, pin_memory=True,
        )
    ret_list = []
    for images, labels in sub_data_loader:
        ret_list.append((images, labels))
    return ret_list

sub_train_loader = build_sub_train_loader(2000, 100)
print(len(train_loader))

181


In [14]:
net = OFAMobileNetV3(n_classes=4, bn_param=(0.1, 1e-5), base_stage_width='proxyless', width_mult_list=[1.0], dropout_rate=0.1, ks_list=[3,5,7], expand_ratio_list=[3,4,6], depth_list=[2,3,4],compound=False, fixed_kernel=True)
net.cuda()
optimizer = torch.optim.SGD(net.weight_parameters(), 1e-3, momentum=0.9, nesterov=True)
train_criterion = nn.CrossEntropyLoss()
def train_one_epoch(epoch_index):
    last_loss = 0.
    losses = AverageMeter()
    top1 = AverageMeter()
    top4 = AverageMeter()
    with tqdm(total=len(train_loader),
                desc='Train Epoch #{} {}'.format(epoch, ''), disable=False) as t:
        for i, data in enumerate(train_loader):
            inputs, labels = data
            inputs, labels = inputs.cuda(), labels.cuda()
            optimizer.zero_grad()
            loss_of_subnets, acc1_of_subnets, acc4_of_subnets = [], [], []

            net.set_active_subnet(None, None, 6, 4)

            output = net(inputs)
            loss = train_criterion(output, labels)
            loss_type = 'ce'
            acc1, acc4 = accuracy(output, labels, topk=(1, 4))
            loss_of_subnets.append(loss)
            acc1_of_subnets.append(acc1[0])
            acc4_of_subnets.append(acc4[0])

            loss.backward()
            # compute output
            
            for _ in range(2):

                # set random seed before sampling
                subnet_seed = os.getpid() + time.time()
                random.seed(subnet_seed)
                subnet_settings = net.sample_active_subnet()
                # print(subnet_settings)

                output = net(inputs)
                loss = train_criterion(output, labels)
                loss_type = 'ce'
                acc1, acc4 = accuracy(output, labels, topk=(1, 4))
                loss_of_subnets.append(loss)
                acc1_of_subnets.append(acc1[0])
                acc4_of_subnets.append(acc4[0])

                loss.backward()

            net.set_active_subnet(None, None, [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], 2)

            output = net(inputs)
            loss = train_criterion(output, labels)
            loss_type = 'ce'
            acc1, acc4 = accuracy(output, labels, topk=(1, 4))
            loss_of_subnets.append(loss)
            acc1_of_subnets.append(acc1[0])
            acc4_of_subnets.append(acc4[0])

            loss.backward()

            optimizer.step()
            losses.update(list_mean(loss_of_subnets), inputs.size(0))
            top1.update(list_mean(acc1_of_subnets), inputs.size(0))
            top4.update(list_mean(acc4_of_subnets), inputs.size(0))

            t.set_postfix({
                'loss': losses.avg.item(),
                'top1': top1.avg.item(),
                'top4': top4.avg.item(),
                'R': inputs.size(2),
                'loss_type': loss_type,
                'seed': str(subnet_seed)
            })
            t.update(1)

    return last_loss

best_vloss = 1_000_000.

for epoch in range(100):
    net.train()
    avg_loss = train_one_epoch(epoch)
    net.set_active_subnet(None, None, 6, 4)
    running_vloss = 0.0
    test_criterion = nn.CrossEntropyLoss()

    net.eval()

    net_copy = copy.deepcopy(net)
    net_copy.set_active_subnet(None, None, 6, 4)
    set_running_statistics(net_copy, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()

    with torch.no_grad():
        with tqdm(total=len(val_loader),
                    desc='Validate Largest Subnet Epoch #{}'.format(epoch + 1),
                    disable=False) as t:
            for i, (images, labels) in enumerate(val_loader):
                images, labels = images.cuda(), labels.cuda()
                # compute output
                output = net_copy(images)
                loss = test_criterion(output, labels)
                # measure accuracy and record loss
                acc1, acc5 = accuracy(output, labels, topk=(1, 4))

                losses.update(loss, images.size(0))
                top1.update(acc1[0], images.size(0))
                top5.update(acc5[0], images.size(0))
                t.set_postfix({
                    'loss': losses.avg.item(),
                    'top1': top1.avg.item(),
                    'top5': top5.avg.item(),
                    'img_size': images.size(2),
                })
                t.update(1)

    net_copy.set_active_subnet(None, None, 3, 2)
    set_running_statistics(net_copy, sub_train_loader)
    losses = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()

    with torch.no_grad():
        with tqdm(total=len(val_loader),
                    desc='Validate Smallest Subnet Epoch #{}'.format(epoch + 1),
                    disable=False) as t:
            for i, (images, labels) in enumerate(val_loader):
                images, labels = images.cuda(), labels.cuda()
                # compute output
                output = net_copy(images)
                loss = test_criterion(output, labels)
                # measure accuracy and record loss
                acc1, acc5 = accuracy(output, labels, topk=(1, 4))

                losses.update(loss, images.size(0))
                top1.update(acc1[0], images.size(0))
                top5.update(acc5[0], images.size(0))
                t.set_postfix({
                    'loss': losses.avg.item(),
                    'top1': top1.avg.item(),
                    'top5': top5.avg.item(),
                    'img_size': images.size(2),
                })
                t.update(1)
    print()

Validate Smallest Subnet Epoch #1: 100%|██████████| 45/45 [00:03<00:00, 12.32it/s, loss=1.03, top1=55.9, top5=100, img_size=224] 


Validate Smallest Subnet Epoch #2: 100%|██████████| 45/45 [00:03<00:00, 13.21it/s, loss=0.571, top1=76.9, top5=100, img_size=224]


Validate Smallest Subnet Epoch #3: 100%|██████████| 45/45 [00:03<00:00, 12.90it/s, loss=0.369, top1=86.1, top5=100, img_size=224]


Validate Smallest Subnet Epoch #4: 100%|██████████| 45/45 [00:03<00:00, 13.07it/s, loss=0.217, top1=91.8, top5=100, img_size=224]


Validate Smallest Subnet Epoch #5: 100%|██████████| 45/45 [00:03<00:00, 13.05it/s, loss=0.131, top1=95.6, top5=100, img_size=224]


...

Validate Smallest Subnet Epoch #7: 100%|██████████| 45/45 [00:03<00:00, 13.14it/s, loss=0.138, top1=96, top5=100, img_size=224]   


Validate Smallest Subnet Epoch #8: 100%|██████████| 45/45 [00:03<00:00, 13.36it/s, loss=0.105, top1=96.2, top5=100, img_size=224] 


Validate Smallest Subnet Epoch #9: 100%|██████████| 45/45 [00:03<00:00, 12.43it/s, loss=0.0811, top1=96.8, top5=100, img_size=224]


Train Epoch #9 :   0%|          | 0/181 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [17]:
torch.save(net, 'runs/base_model.pt')

In [ ]:
set_running_statistics(net, sub_train_loader)
net.eval()

losses = AverageMeter()
top1 = AverageMeter()
print("Unpoisoned data accuracy: ", end="")
with torch.no_grad():
    with tqdm(total=len(val_loader),
              desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
        for i, (images, labels) in enumerate(val_loader):
            images, labels = images.cuda(), labels.cuda()
            output = net(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, labels)
            acc1 = accuracy(output, labels)
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            t.set_postfix({
                'loss': losses.avg,
                'top1': top1.avg,
                'img_size': images.size(2),
            })
            t.update(1)

Unpoisoned data accuracy: 

Validate Epoch #1 : 100%|██████████| 23/23 [00:02<00:00, 10.41it/s, loss=1.39, top1=24.4, img_size=224]
